# PyTestLab – Notebook GUI Demo

This short notebook shows how you can build **interactive controls** for *any* PyTestLab instrument **without writing specialized widget classes** – everything is declared in a few lines.

*The demo runs entirely in simulation mode – no hardware required.*

In [ ]:
# Install ipywidgets if it's missing.  (Google Colab already has it; classic Jupyter may need it once.)
import importlib.util
import subprocess
import sys
import warnings

import ipywidgets as w

if importlib.util.find_spec("ipywidgets") is None:
    warnings.warn("Installing ipywidgets…", stacklevel=2)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets>=8"])

# Optional – nice layout in classic Notebook
try:
    w.install_nbextension(enable=True)
except Exception:
    pass

In [ ]:
# 1. Create (simulated) instruments
from pytestlab import AutoInstrument


def main():
    # Create simulated instruments using current PyTestLab patterns
    psu = AutoInstrument.from_config("keysight/EDU36311A", simulate=True)
    psu.connect_backend()
    
    dmm = AutoInstrument.from_config("keysight/34470A", simulate=True) 
    dmm.connect_backend()
    
    awg = AutoInstrument.from_config("keysight/EDU33212A", simulate=True)
    awg.connect_backend()
    
    print("Simulated PSU, DMM & AWG ready → build GUI panels below…")
    return psu, dmm, awg

# Create instruments
psu, dmm, awg = main()

In [ ]:
# 2. Build a PSU control panel (Channel 1)
from pytestlab.gui.builder import Button
from pytestlab.gui.builder import InstrumentPanel
from pytestlab.gui.builder import Slider
from pytestlab.gui.builder import Toggle

# Use chainable facade API as shown in README
psu_panel = InstrumentPanel(
    title="PSU – CH1 (Sim)",
    inst=psu,
    controls=[
        Slider(
            label="Voltage [V]",
            getter=lambda i: i.channel(1).get_voltage(),
            setter=lambda i, v: i.channel(1).set_voltage(v),
            min=0.0, max=6.0, step=0.05,
        ),
        Slider(
            label="Current Limit [A]",
            getter=lambda i: i.channel(1).get_current_limit(),
            setter=lambda i, c: i.channel(1).set_current_limit(c),
            min=0.0, max=5.0, step=0.05,
        ),
        Toggle(
            label="Output",
            getter=lambda i: i.channel(1).is_output_enabled(),
            setter=lambda i, state: i.channel(1).on() if state else i.channel(1).off(),
            on_desc="ON", off_desc="OFF",
        ),
        Button(
            label="⟳ Refresh read-back",
            action=lambda i, p: p.refresh(),
        ),
    ],
)

# Move the slider, toggle the output – the PSU simulation follows immediately.
psu_panel

In [ ]:
# 3. Build a DMM one-click read-out
dmm_panel = InstrumentPanel(
    title="DMM – DC Voltage (Sim)",
    inst=dmm,
    controls=[
        Button(
            label="Measure DC Voltage",
            action=lambda dev, p: print("Measured:", dev.measure_voltage_dc()),
            style="success",
        ),
        Button(
            label="Measure AC Voltage", 
            action=lambda dev, p: print("Measured:", dev.measure_voltage_ac()),
            style="info",
        ),
        Button(
            label="Measure Current",
            action=lambda dev, p: print("Measured:", dev.measure_current_dc()),
            style="warning",
        ),
    ],
)

# Click **Measure** buttons – the values are printed using the current API
dmm_panel

In [ ]:
# 4. Build an AWG control panel using current patterns
awg_panel = InstrumentPanel(
    title="AWG – CH1 (Sim)",
    inst=awg,
    controls=[
        Slider(
            label="Frequency [Hz]",
            getter=lambda i: i.channel(1).get_frequency(),
            setter=lambda i, f: i.channel(1).setup_sine(frequency=f, amplitude=1.0),
            min=1.0, max=1000000.0, step=100.0,
        ),
        Slider(
            label="Amplitude [V]",
            getter=lambda i: i.channel(1).get_amplitude(),
            setter=lambda i, a: i.channel(1).setup_sine(frequency=1000.0, amplitude=a),
            min=0.01, max=5.0, step=0.01,
        ),
        Toggle(
            label="Output",
            getter=lambda i: i.channel(1).is_output_enabled(),
            setter=lambda i, state: i.channel(1).enable() if state else i.channel(1).disable(),
            on_desc="ON", off_desc="OFF",
        ),
        Button(
            label="Generate Sine Wave",
            action=lambda i, p: i.channel(1).setup_sine(frequency=1000.0, amplitude=1.0),
            style="primary",
        ),
    ],
)

# Control AWG parameters with the modern chainable API
awg_panel

## Where to go next?

* **Add more controls** – Use `Slider`, `Toggle` or `Button` controls for any instrument parameter. Everything is declarative.
* **Chainable Facade API** – Notice how we use `instrument.channel(1).set_voltage(5.0).on()` for clean, readable control.
* **Bench Integration** – Load instruments from `bench.yaml` files for real lab setups:
  ```python
  import pytestlab
  with pytestlab.Bench.open("bench.yaml") as bench:
      # Use bench.psu, bench.dmm, etc. in your GUI panels
  ```
* **MeasurementSession** – Combine GUI controls with automated parameter sweeps and data logging.
* **Record & Replay** – Record real instrument interactions and replay them exactly for reproducible measurements.
* **Live Hardware** – Replace `simulate=True` with real VISA addresses and you have a *live* lab dashboard!

See the README.md and `examples/` directory for more advanced usage patterns.